# Coverage-Based Scene Selection

This notebook demonstrates the full workflow for finding cloud-free optical
scenes that cover a bounding box:

1. **Search ItsLive STAC** for optical velocity pairs that overlap the bbox.
2. **Score (date, WRS-path) groups** — for each pair, download the minimal
   ItsLive NetCDF coverage layer (`chip_size_height`). Accumulate the
   per-cell *maximum* coverage across all pairs a scene participates in,
   then union across same-day / same-path scenes.
3. **Inspect the ranked groups** — pairs and coverage fractions.
4. **Download and mosaic** the best-covering group into a single GeoTIFF
   per band, in EPSG:3031.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import rasterio

from itslive_cloudfree.stac_client import search_granules
from itslive_cloudfree.coverage import score_date_path_groups
from itslive_cloudfree.downloader import download_scenes_mosaic

## 1 — Define the search parameters

The bbox is in **EPSG:3031** (Antarctic Polar Stereographic, metres).
Working in the native polar CRS avoids the distorted footprints that arise
from rectangular WGS-84 bboxes at polar latitudes.

In [ ]:
# Pine Island Glacier — bbox in EPSG:3031 (metres)
BBOX     = (-1_660_000, -365_000, -1_490_000, -235_000)
BBOX_CRS = 3031

START_DATE = "2019-10-01"
END_DATE   = "2020-03-31"

OUT_DIR = Path("coverage_output")
OUT_DIR.mkdir(exist_ok=True)

COOKIE_FILE = str(Path.home() / ".usgs_landsat_cookies.txt")

## 2 — Search ItsLive for optical pairs

In [ ]:
print("Searching ItsLive STAC…")
granules = list(search_granules(
    bbox=BBOX,
    start_date=START_DATE,
    end_date=END_DATE,
    bbox_crs=BBOX_CRS,
    optical_only=True,
))
print(f"Found {len(granules)} optical granule(s).")

## 3 — Score (date, WRS-path) groups by bbox coverage

`score_date_path_groups` downloads the `chip_size_height` variable from
each ItsLive NetCDF (~5 MB each) and builds a 120 m boolean coverage grid
over the bbox.  Groups are ranked by the fraction of bbox cells with at
least one valid ItsLive match.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

groups = score_date_path_groups(
    granules,
    bbox=BBOX,
    bbox_crs=BBOX_CRS,
    granule_crs=3031,
)

print(f"\nFound {len(groups)} (date, path) group(s).\n")
print(f"{'Date':<12} {'Path':<6} {'Coverage':>10}  Scenes")
print("-" * 60)
for g in groups[:20]:
    scene_ids = ", ".join(s.scene_id for s in g.scenes)
    print(f"{str(g.acquisition_date):<12} {g.wrs_path:<6} {g.coverage_fraction:>9.1%}  {scene_ids}")

## 4 — Visualise coverage grids for the top groups

In [ ]:
top_n = min(6, len(groups))
if top_n == 0:
    print("No groups found — nothing to plot.")
else:
    ncols = min(3, top_n)
    nrows = (top_n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
    axes = np.array(axes).flatten()

    for i, g in enumerate(groups[:top_n]):
        ax = axes[i]
        ax.imshow(g.coverage_grid, cmap="Blues", origin="upper", vmin=0, vmax=1)
        ax.set_title(
            f"{g.acquisition_date}  path {g.wrs_path}\n"
            f"coverage {g.coverage_fraction:.1%}",
            fontsize=9,
        )
        ax.axis("off")

    for j in range(top_n, len(axes)):
        axes[j].axis("off")

    fig.suptitle("ItsLive match coverage over bbox (blue = covered)", fontsize=11)
    plt.tight_layout()
    plt.show()

## 5 — Download and mosaic the best group

`download_scenes_mosaic` resolves the STAC asset URLs for all scenes in the
group, then calls `gdal.Warp()` with the full list of COG hrefs.  GDAL builds
an internal VRT mosaic and reprojects to EPSG:3031 in a single pass.

In [ ]:
if not groups:
    print("No groups — skipping download.")
else:
    best = groups[0]
    print(f"Downloading best group: {best.acquisition_date}  path {best.wrs_path}  "
          f"coverage {best.coverage_fraction:.1%}")
    print(f"Scenes: {[s.scene_id for s in best.scenes]}")

    cookie_path = Path(COOKIE_FILE)
    env_kwargs = {}
    if cookie_path.exists():
        env_kwargs = dict(GDAL_HTTP_COOKIEFILE=COOKIE_FILE, GDAL_HTTP_COOKIEJAR=COOKIE_FILE)

    with rasterio.Env(**env_kwargs):
        mosaic_paths = download_scenes_mosaic(
            best.scenes,
            bbox=BBOX,
            bbox_crs=BBOX_CRS,
            dst_crs=3031,
            out_dir=OUT_DIR,
        )

    print("\nOutput files:")
    for band, path in mosaic_paths.items():
        with rasterio.open(path) as src:
            print(f"  {band}: {src.width}×{src.height} px  CRS=EPSG:{src.crs.to_epsg()}  → {path.name}")

## 6 — Quick look at the downloaded bands

In [ ]:
if not groups or not mosaic_paths:
    print("No mosaic to display.")
else:
    fig, axes = plt.subplots(1, len(mosaic_paths), figsize=(5 * len(mosaic_paths), 4))
    if len(mosaic_paths) == 1:
        axes = [axes]

    for ax, (band, path) in zip(axes, mosaic_paths.items()):
        with rasterio.open(path) as src:
            data = src.read(1).astype(float)
        data[data == 0] = np.nan
        p2, p98 = np.nanpercentile(data, [2, 98])
        ax.imshow(data, vmin=p2, vmax=p98, cmap="gray")
        ax.set_title(band, fontsize=9)
        ax.axis("off")

    title = (
        f"{best.acquisition_date}  path {best.wrs_path}  "
        f"coverage {best.coverage_fraction:.1%}"
    )
    fig.suptitle(title, fontsize=10)
    plt.tight_layout()
    plt.show()